In [1]:
import sys
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR     = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "processed"

BIZ_JSON    = RAW_DIR / "yelp_academic_dataset_business.json"
REVIEW_JSON = RAW_DIR / "yelp_academic_dataset_review.json"
USER_JSON   = RAW_DIR / "yelp_academic_dataset_user.json"

print("Raw files:")
for f in [BIZ_JSON, REVIEW_JSON, USER_JSON]:
    size_mb = f.stat().st_size / 1_048_576
    with open(f) as fp:
        n = sum(1 for _ in fp)
    print(f"  {f.name:<50} {size_mb:>7,.0f} MB  |  {n:>10,} records")

Raw files:
  yelp_academic_dataset_business.json                    113 MB  |     150,346 records


  yelp_academic_dataset_review.json                    5,094 MB  |   6,990,280 records


  yelp_academic_dataset_user.json                      3,208 MB  |   1,987,897 records


In [2]:
from src.ingestion import load_businesses

businesses = load_businesses(BIZ_JSON)

print(f"\nFiltered business dataset: {len(businesses):,} locations")

Loading businesses from: yelp_academic_dataset_business.json


  Scanned 150,346 businesses → kept 8,203 matching records

Filtered business dataset: 8,203 locations


In [3]:
# Brand composition
starbucks = businesses["name"].str.contains("Starbucks", case=False, na=False).sum()
dunkin    = businesses["name"].str.contains("Dunkin", case=False, na=False).sum()
others    = len(businesses) - starbucks - dunkin

print("Brand composition:")
print(f"  Starbucks (primary subject)      : {starbucks:>5,}")
print(f"  Dunkin' (direct competitor)      : {dunkin:>5,}")
print(f"  All other coffee businesses      : {others:>5,}")
print(f"  Total                            : {len(businesses):>5,}")

Brand composition:
  Starbucks (primary subject)      :   683
  Dunkin' (direct competitor)      :   550
  All other coffee businesses      : 6,970
  Total                            : 8,203


In [4]:
# Geographic distribution
print("Top 15 states by location count:")
print(businesses["state"].value_counts().head(15).to_string())

Top 15 states by location count:
state
PA    2101
FL    1502
LA     681
TN     654
IN     626
MO     586
NJ     471
AZ     458
NV     337
ID     270
CA     258
DE     148
IL     111


In [5]:
from src.ingestion import load_reviews_chunked

DATE_START = "2017-01-01"
DATE_END   = "2021-12-31"

reviews = load_reviews_chunked(
    review_json_path=REVIEW_JSON,
    business_ids=set(businesses["business_id"]),
    date_start=DATE_START,
    date_end=DATE_END,
)

print(f"\nReview dataset: {len(reviews):,} records")

Streaming reviews from: yelp_academic_dataset_review.json
  Date window: 2017-01-01 → 2021-12-31
  Filtering against 8,203 business IDs


  ... scanned 500,000 lines, collected 26,532 matching reviews


  ... scanned 1,000,000 lines, collected 52,724 matching reviews


  ... scanned 1,500,000 lines, collected 83,812 matching reviews


  ... scanned 2,000,000 lines, collected 108,058 matching reviews


...

  ... scanned 5,000,000 lines, collected 266,823 matching reviews


  ... scanned 5,500,000 lines, collected 296,175 matching reviews


  ... scanned 6,000,000 lines, collected 320,548 matching reviews


  ... scanned 6,500,000 lines, collected 349,712 matching reviews


  Scanned 6,990,280 reviews → kept 381,999 matching records

Review dataset: 381,999 records


In [6]:
from src.ingestion import load_users_subset

users = load_users_subset(
    user_json_path=USER_JSON,
    user_ids=set(reviews["user_id"]),
)

print(f"User profiles loaded: {len(users):,}")

Loading user profiles from: yelp_academic_dataset_user.json
  Looking up 241,363 user IDs


  Scanned 1,987,895 user records → matched 241,363
User profiles loaded: 241,363


In [7]:
print("=" * 55)
print("DATASET SUMMARY")
print("=" * 55)
print(f"  Geography    : United States")
print(f"  Time window  : {DATE_START} → {DATE_END}")
print(f"  Businesses   : {len(businesses):,}")
print(f"  Reviews      : {len(reviews):,}")
print(f"  Users        : {len(users):,}")
print(f"  Avg stars    : {reviews['review_stars'].mean():.2f}")
print(f"  Date range   : {reviews['date'].min()} → {reviews['date'].max()}")
print("=" * 55)

DATASET SUMMARY
  Geography    : United States
  Time window  : 2017-01-01 → 2021-12-31
  Businesses   : 8,203
  Reviews      : 381,999
  Users        : 241,363
  Avg stars    : 3.94
  Date range   : 2017-01-01 → 2021-12-31


In [8]:
# Reviews table — structure
print(f"reviews: {reviews.shape[0]:,} rows × {reviews.shape[1]} columns\n")
print(reviews.dtypes)
print("\nNull counts:")
print(reviews.isnull().sum())

reviews: 381,999 rows × 9 columns

review_id        object
user_id          object
business_id      object
review_stars    float64
date             object
text             object
useful            int64
funny             int64
cool              int64
dtype: object

Null counts:
review_id       0
user_id         0
business_id     0
review_stars    0
date            0
text            0
useful          0
funny           0
cool            0
dtype: int64


In [9]:
# Reviews — first 5 rows
for i, row in reviews.head(5).iterrows():
    print(f"--- Row {i} ---")
    for col in reviews.columns:
        val = str(row[col])
        if col == "text": val = val[:100]
        print(f"  {col}: {val}")
    print()

--- Row 0 ---
  review_id: -P5E9BYUaK7s3PwBF5oAyg
  user_id: Jha0USGDMefGFRLik_xFQg
  business_id: bMratNjTG5ZFEA6hVyr-xQ
  review_stars: 5.0
  date: 2017-02-19
  text: First time there and it was excellent!!! It feels like your are entering someone's home. The waiters
  useful: 0
  funny: 0
  cool: 0

--- Row 1 ---
  review_id: A4n4YaE-owOVgTQcrVqHUw
  user_id: S7bjj-L07JuRr-tpX1UZLw
  business_id: I6L0Zxi5Ww0zEWSAVgngeQ
  review_stars: 4.0
  date: 2018-07-07
  text: The cafe was extremely cute. We came at 8am and they even had a jazz band playing at that time. I go
  useful: 0
  funny: 0
  cool: 0

--- Row 2 ---
  review_id: r1tPwFMILy0COeEQ-B3YLw
  user_id: 3M1_pyDSgMP6sRMz564eJw
  business_id: 8xM8akbQhHDQdJO1sPMB1A
  review_stars: 5.0
  date: 2018-07-26
  text: I had the pleasure to meet with Ann today and not only was she extremely helpful and so positive, sh
  useful: 0
  funny: 0
  cool: 0

--- Row 3 ---
  review_id: wsFRDsHxz2mM_Ettgn1qQg
  user_id: x8ErSBur0SsnL1lZwP5o4Q
  bu

In [10]:
# Businesses table — structure
print(f"businesses: {businesses.shape[0]:,} rows × {businesses.shape[1]} columns\n")
print(businesses.dtypes)
print("\nNull counts:")
print(businesses.isnull().sum())

businesses: 8,203 rows × 7 columns

business_id               object
name                      object
city                      object
state                     object
categories                object
business_avg_stars       float64
business_review_count      int64
dtype: object

Null counts:
business_id              0
name                     0
city                     0
state                    0
categories               0
business_avg_stars       0
business_review_count    0
dtype: int64


In [11]:
# Businesses — first 5 rows
for i, row in businesses.head(5).iterrows():
    print(f"--- Row {i} ---")
    for col in businesses.columns:
        print(f"  {col}: {row[col]}")
    print()

--- Row 0 ---
  business_id: mWMc6_wTdE0EUBKIGXDVfA
  name: Perkiomen Valley Brewery
  city: Green Lane
  state: PA
  categories: Brewpubs, Breweries, Food
  business_avg_stars: 4.5
  business_review_count: 13

--- Row 1 ---
  business_id: ljxNT9p0y7YMPx0fcNBGig
  name: Tony's Restaurant & 3rd Street Cafe
  city: Alton
  state: IL
  categories: Restaurants, Specialty Food, Steakhouses, Food, Italian, Pizza, Pasta Shops
  business_avg_stars: 3.0
  business_review_count: 94

--- Row 2 ---
  business_id: lk9IwjZXqUMqqOhM774DtQ
  name: Caviar & Bananas
  city: Nashville
  state: TN
  categories: Coffee & Tea, Restaurants, Wine Bars, Bars, Nightlife, American (Traditional), Event Planning & Services, Food, Caterers, Breakfast & Brunch, Cafes, Diners
  business_avg_stars: 3.5
  business_review_count: 159

--- Row 3 ---
  business_id: 7clCBgNbd-x2Wj96lZ6Mjw
  name: Bier Brewery and Tap Room
  city: Indianapolis
  state: IN
  categories: Food, Beer, Wine & Spirits, Breweries
  business_avg_sta

In [12]:
# Star rating distribution
print("Star rating distribution:")
star_dist = reviews["review_stars"].value_counts().sort_index()
for stars, count in star_dist.items():
    bar = "█" * int(count * 40 / star_dist.max())
    pct = count / len(reviews) * 100
    print(f"  {int(stars)}★  {bar:<40}  {count:>8,}  ({pct:.1f}%)")

Star rating distribution:
  1★  █████████                                   45,497  (11.9%)
  2★  █████                                       27,379  (7.2%)
  3★  ██████                                      34,752  (9.1%)
  4★  ██████████████                              72,820  (19.1%)
  5★  ████████████████████████████████████████   201,551  (52.8%)


In [13]:
businesses.to_parquet(INTERIM_DIR / "businesses_filtered.parquet", index=False)
reviews.to_parquet(INTERIM_DIR / "reviews_filtered.parquet",       index=False)
users.to_parquet(INTERIM_DIR / "users_filtered.parquet",           index=False)

with open(INTERIM_DIR / "analysis_config.json", "w") as f:
    json.dump({"date_start": DATE_START, "date_end": DATE_END}, f, indent=2)

print("Files saved to data/processed/:")
for f in INTERIM_DIR.iterdir():
    print(f"  {f.name:<45} {f.stat().st_size / 1_048_576:>6.1f} MB")

Files saved to data/processed/:
  reviews_filtered.parquet                       131.0 MB
  analysis_config.json                             0.0 MB
  analysis_ready.parquet                         141.7 MB
  users_filtered.parquet                          10.1 MB
  businesses_filtered.parquet                      0.5 MB
  cleaned_joined.parquet                         139.0 MB
